# 🎞️ ROTBOTS — Assemble
## Videos + Effects + Audio + Music + Subtitles + Credits → Final

---

All features toggleable. Effects from 03_Effects_and_Log are auto-detected.

> **🤔** How does editing, effects, music, and subtitles change meaning?

---

In [ ]:
# SETUP
import json, subprocess, shutil, os, re
from pathlib import Path
from IPython.display import display, Markdown, Video
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)
print('✅ Setup')

In [ ]:
# SELECT SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'session_info.json').exists()])
if not sessions: print('❌ No sessions!')
else:
    print('📂 Sessions:')
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
VIDEOS_DIR = SESSION_DIR / 'videos'
AUDIO_DIR = SESSION_DIR / 'audio'
sub_file = SESSION_DIR / 'subtitles.ass'
video_files = sorted(VIDEOS_DIR.glob('scene_*.mp4')) if VIDEOS_DIR.exists() else []
audio_files = sorted(AUDIO_DIR.glob('narration_*.mp3')) if AUDIO_DIR.exists() else []
essay = None
if (SESSION_DIR/'essay_script.json').exists():
    with open(SESSION_DIR/'essay_script.json') as f: essay = json.load(f)
# Load prompts for effects
prompts_data = []
if (SESSION_DIR/'prompts.json').exists():
    with open(SESSION_DIR/'prompts.json') as f: prompts_data = json.load(f)
effects_map = {p['scene']: p for p in prompts_data if p.get('ffmpeg_effects')}
print(f'✅ Session: {SESSION_NAME}')
print(f'   🎬 {len(video_files)} videos | 🎙️ {len(audio_files)} audio | 📝 Subs: {"✅" if sub_file.exists() else "—"} | 🎨 Effects: {len(effects_map)} scenes')
if not video_files: print('❌ No videos!')

In [ ]:
# OPTIONS
ENABLE_NARRATION = True
ENABLE_MUSIC = False
ENABLE_SUBTITLES = False
ENABLE_CREDITS = True
MUSIC_SOURCE = 'generate'
MUSIC_PROMPT = 'Ambient cinematic background music, slow contemplative mood'
MUSIC_VOLUME = 0.10

print('🎬 Options:')
for name,val in [('Narration',ENABLE_NARRATION),('Music',ENABLE_MUSIC),('Subtitles',ENABLE_SUBTITLES),('Credits',ENABLE_CREDITS)]:
    print(f'   {name}: {"✅" if val else "❌"}')

In [ ]:
# HELPERS
def dur(p):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0
def ff(cmd,desc=''):
    if desc: print(f'   {desc}...',end=' ',flush=True)
    r=subprocess.run(cmd,capture_output=True,text=True,timeout=600)
    if r.returncode==0:
        if desc: print('✅')
        return True
    print(f'❌ {r.stderr[:300]}'); return False

def build_effect_filter(name, intensity=0.7):
    """Build FFmpeg filter for a named effect (from original ROTBOTS)."""
    i = max(0.0, min(1.0, intensity))
    if name == 'film_grain': return f'noise=alls={int(12*i)}:allf=t'
    if name == 'vhs_artifacts': return f'noise=alls={int(30*i)}:allf=t,rgbashift=rh={int(2*i)}:bh={int(-2*i)},eq=contrast={1+0.1*i:.3f}:brightness={0.02*i:.3f}:saturation={1-0.15*i:.3f}'
    if name == 'celluloid_scratches': return f'noise=c0s={int(15*i)}:c0f=t:c1s={int(15*i)}:c1f=t,eq=contrast={1+0.05*i:.3f}:brightness={-0.01*i:.3f}'
    if name == 'sepia_tone':
        inv=1-i; return f'colorchannelmixer={inv+i*0.393:.3f}:{i*0.769:.3f}:{i*0.189:.3f}:0:{i*0.349:.3f}:{inv+i*0.686:.3f}:{i*0.168:.3f}:0:{i*0.272:.3f}:{i*0.534:.3f}:{inv+i*0.131:.3f}:0'
    if name == 'bw_transition':
        inv=1-i; return f'colorchannelmixer={inv+i*0.3:.3f}:{i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{inv+i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{i*0.4:.3f}:{inv+i*0.3:.3f}:0'
    if name == 'color_grade_warm': return f'eq=saturation={1+0.1*i:.3f}:brightness={0.02*i:.3f},colorbalance=rs={0.1*i:.3f}:gs={0.05*i:.3f}:bs={-0.05*i:.3f}'
    if name == 'color_grade_cool': return f'eq=saturation={1-0.1*i:.3f},colorbalance=rs={-0.05*i:.3f}:gs=0:bs={0.12*i:.3f}'
    if name == 'vignette': return f'vignette=PI/4*{i:.3f}'
    if name == 'flicker': return f"noise=alls={int(8*i)}:allf=t,eq=brightness='{0.02*i:.4f}*sin(2*PI*t*3)'"
    if name == 'desaturate': return f'eq=saturation={0.5+0.5*(1-i):.3f}'
    return None

In [ ]:
# STEP 1: NORMALIZE + APPLY EFFECTS
print('🔧 Step 1: Normalizing + applying effects...')
norm = []
for v in video_files:
    out = TEMP / v.name
    # Extract scene number from filename
    m = re.search(r'scene_(\d+)', v.name)
    sn = int(m.group(1)) if m else 0
    
    # Base normalize filter
    vf = 'scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black'
    
    # Add effects if assigned
    if sn in effects_map:
        p = effects_map[sn]
        eff_names = p.get('ffmpeg_effects', [])
        intensity = p.get('effect_intensity', 0.7)
        for eff in eff_names:
            f = build_effect_filter(eff, intensity)
            if f: vf += ',' + f
        eff_tag = f' + {eff_names}'
    else:
        eff_tag = ''
    
    if ff(['ffmpeg','-y','-i',str(v),'-vf',vf,'-r','24','-pix_fmt','yuv420p','-c:v','libx264','-preset','fast','-crf','23','-an',str(out)], f'{v.name}{eff_tag}'):
        norm.append(out)
print(f'✅ {len(norm)} videos processed')

In [ ]:
# STEP 2: MUSIC
music_path = None
if ENABLE_MUSIC:
    if MUSIC_SOURCE == 'generate':
        print('🎵 Step 2: Generating music...')
        try:
            import fal_client, time, requests
            if not os.environ.get('FAL_KEY'):
                FAL_KEY = ''
                if FAL_KEY: os.environ['FAL_KEY'] = FAL_KEY
                else: raise Exception('Set FAL_KEY or run 05_Generate first')
            td = sum(dur(v) for v in norm) + 10
            t0 = time.time()
            result = fal_client.subscribe('beatoven/music-generation', arguments={'prompt':MUSIC_PROMPT,'duration':min(150,int(td))})
            url = None
            if isinstance(result,dict):
                url = result.get('audio',{}).get('url') if isinstance(result.get('audio'),dict) else result.get('audio') or result.get('url')
            if url:
                music_path = TEMP / 'music.wav'
                with open(music_path,'wb') as f: f.write(requests.get(url,timeout=120).content)
                print(f'   ✅ ({time.time()-t0:.0f}s)')
            else: print('   ⚠️ No URL')
        except Exception as e: print(f'   ❌ {e}')
    elif MUSIC_SOURCE == 'upload':
        from google.colab import files
        for fn,content in files.upload().items():
            music_path = TEMP/'music.mp3'
            with open(music_path,'wb') as f: f.write(content); break
else: print('🎵 Step 2: Music — skipped')

In [ ]:
# STEP 3: CREDITS
credits_path = None
if ENABLE_CREDITS:
    print('📜 Step 3: Credits...')
    title = essay.get('title','Untitled') if essay else 'Untitled'
    sources = essay.get('credits',{}).get('sources',[]) if essay else []
    lines = [title,'','Generated by ROTBOTS','','— Sources —']
    for s in sources: lines.append(s[:60]+'...' if len(s)>60 else s)
    lines += ['','— Pipeline —','LLM: Groq','Video: fal.ai','Voice: Edge-TTS','FFmpeg','','LI-MA TDA 2026']
    D=8;LH=42;spd=(720+len(lines)*LH)/D
    flt=[f"drawtext=text='{l.replace(chr(39),chr(8217)).replace(chr(58),chr(92)+chr(58))}':fontsize=28:fontcolor=white:x=(w-text_w)/2:y=h+{i*LH}-{spd:.1f}*t" for i,l in enumerate(lines) if l]
    credits_path=TEMP/'credits.mp4'
    ff(['ffmpeg','-y','-f','lavfi','-i',f'color=c=black:s=1280x720:d={D}:r=24','-vf',','.join(flt),'-pix_fmt','yuv420p','-c:v','libx264','-preset','fast',str(credits_path)],'Credits')
else: print('📜 Step 3: Credits — skipped')

In [ ]:
# STEP 4: CONCATENATE
print('🎬 Step 4: Concat...')
cf=TEMP/'concat.txt'
with open(cf,'w') as f:
    for v in norm: f.write(f"file '{v}'\n")
    if credits_path and credits_path.exists(): f.write(f"file '{credits_path}'\n")
concat_out=TEMP/'concatenated.mp4'
ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(cf),'-c','copy',str(concat_out)],'Concat')
print(f'   {dur(concat_out):.1f}s')

In [ ]:
# STEP 5: AUDIO
audio_out=TEMP/'with_audio.mp4'
has_narr=ENABLE_NARRATION and audio_files
has_mus=music_path and music_path.exists()
if has_narr:
    print('🎙️ Step 5: Audio...')
    acf=TEMP/'ac.txt'
    with open(acf,'w') as f:
        for a in audio_files: f.write(f"file '{a}'\n")
    narr=TEMP/'narration.mp3'
    ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(acf),'-c','copy',str(narr)],'Narration')
    if has_mus:
        mx=TEMP/'mixed.mp3'
        ff(['ffmpeg','-y','-i',str(narr),'-i',str(music_path),'-filter_complex',f'[0:a]volume=1.0[n];[1:a]volume={MUSIC_VOLUME}[m];[n][m]amix=inputs=2:duration=longest[out]','-map','[out]','-c:a','libmp3lame',str(mx)],'Mix')
        ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(mx),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-shortest',str(audio_out)],'Add audio')
    else:
        ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(narr),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-shortest',str(audio_out)],'Add narration')
elif has_mus:
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(music_path),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-shortest',str(audio_out)],'Music only')
else: shutil.copy2(concat_out,audio_out)

In [ ]:
# STEP 6: SUBTITLES
final=SESSION_DIR/'final_video.mp4'
if ENABLE_SUBTITLES and sub_file.exists():
    print('📝 Step 6: Subtitles...')
    la=TEMP/'subtitles.ass'; shutil.copy2(sub_file,la)
    ff(['ffmpeg','-y','-i',str(audio_out),'-vf',f'ass={la}','-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)],'Burn subs')
elif ENABLE_SUBTITLES: print('📝 No subtitles.ass — run 07_Subtitles'); shutil.copy2(audio_out,final)
else: shutil.copy2(audio_out,final)
if final.exists(): print(f'\n✅ Final: {dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')

---
## 🎬 Watch!

In [ ]:
title = essay.get('title','Untitled') if essay else 'Untitled'
if final.exists():
    display(Markdown(f'# 🎬 {title}\n*Generated by ROTBOTS*\n\n---'))
    try: display(Video(str(final),embed=True,width=720))
    except: print(f'Drive: rotbots_workshop/{SESSION_NAME}/final_video.mp4')
else: print('❌')

In [ ]:
DOWNLOAD=False
if DOWNLOAD and final.exists():
    from google.colab import files; files.download(str(final))
else: print(f'📁 Drive: rotbots_workshop/{SESSION_NAME}/final_video.mp4')

---
## 📊 Summary

In [ ]:
print('📊 Pipeline Summary')
print('='*50)
st=[]
if (SESSION_DIR/'summaries.json').exists():
    with open(SESSION_DIR/'summaries.json') as f: d=json.load(f)
    st.append(f'📥 {len(d.get("sources",[]))} sources')
if essay:
    tw=sum(len(s.get('narration','').split()) for c in essay.get('chapters',[]) for s in c.get('sections',[]))
    st.append(f'✍️ "{essay.get("title","")}" ({tw}w)')
st.append(f'🎬 {len(video_files)} clips')
if effects_map: st.append(f'🎨 {len(effects_map)} scenes with effects')
st.append(f'🎙️ {len(audio_files)} narrations')
if ENABLE_MUSIC and music_path: st.append('🎵 Music')
if ENABLE_SUBTITLES and sub_file.exists(): st.append('📝 Subtitles')
if ENABLE_CREDITS: st.append('📜 Credits')
if final.exists(): st.append(f'🎞️ {dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')
for s in st: print(f'   {s}')
print(f'\n🤖 All from one topic.')

---
## 🏁 Done!

sources → essay → prompts → effects → voice → video → music → subtitles → credits → final.

**Every step: automated decisions.** What does that mean?

---
*ROTBOTS — LI-MA TDA 2026, Amsterdam*